# 21 — LSTM sobre ventanas contiguas · E4 · h=5 · corte `main`


Auto-generado por `build_notebook_21_lstm_contiguous.py`. **No editar a mano.**

Reentrena el LSTM sobre la población canónica definida en
`docs/plan-reentrenamiento.md`:

- **C1** — una muestra es `(empresaid, direction, start_ts, horizon)`; el ancla es
  un instante, no un índice de fila, y cada objetivo se emite **una sola vez**.
- **C2** — la ventana solo es válida si sus marcas de tiempo son minutos
  consecutivos, de modo que el objetivo cae exactamente a `h` minutos del final.
- **C3** — sin bandera de día atípico: es un agregado del día completo y por lo
  tanto no se conoce al momento de predecir.

El portón de dígitos verifica que el índice reconstruido acá coincida con
`sample_index_manifest.csv`. El refit del XGBoost corre el mismo portón, así que
ambas familias consumen las mismas muestras **por construcción**, no por un join
posterior.

Los notebooks 11/12/13 y 17/18/19 **no se tocan**: siguen sosteniendo la
comparación entre arquitecturas sobre el pipeline anterior.

In [ ]:

import hashlib

import polars as pl
import numpy as np
from pathlib import Path

# Frozen SHA-256 of every required training input. The run stops BEFORE training
# when a required file is missing or its bytes differ from the pinned snapshot.
# `atypical_days.csv` is deliberately absent: the flag was removed (plan C3).
INPUT_HASHES = {
    "headways_E4.parquet": "1dde7f38eea9bc7d9941c17cbc3d326cb864e70be815a1a7e3d0ae2691f19273"
}

# Frozen digests of the canonical sample index, from sample_index_manifest.csv.
# Recomputing them here is the shared-population gate.
INDEX_DIGESTS = {
    "E4|train": "b3bd4bdfca8b53e5db61e3da86a78d5891ccaf0858a7375035138c94a8cf3808",
    "E4|val": "221d160532ac68f38bcde8986f7625073dc171fe7d24f26d2e42410e1e600722",
    "E4|test": "2a2da36998abd23cb49c96d3a77063f8a618a8adde02554bf4b2eb10190f70ca"
}

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _resolve_input(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    candidates = [p for root in roots if root.exists() for p in sorted(root.rglob(name))]
    if not candidates:
        raise FileNotFoundError(f"Required input not found anywhere: {name}")
    for path in candidates:
        if _sha256_file(path) == INPUT_HASHES[name]:
            return path
    raise ValueError(
        f"No copy of {name} matches its frozen SHA-256 — "
        f"candidates: {[str(p) for p in candidates]}"
    )

# Report-only comparison input (NOT a training input): outside the frozen gate.
def _find_baselines_csv() -> Path | None:
    name = "baselines_E4_results_multih.csv"
    for root in (Path("/kaggle/input"), Path(".")):
        if root.exists():
            found = list(root.rglob(name))
            if found:
                return found[0]
    return None

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
HORIZON = 5
GROUP = "baselines"

# Evaluation origin. Resolved against the embedded `splits` module in the prepare
# cell — this cell runs before the embeds, so only the NAME can live here.
FOLD_NAME = "main"

RESULTS_OUT = OUTPUT_DIR / f"lstm_contig_E4_results_h{HORIZON}.csv"
RESID_OUT = OUTPUT_DIR / f"lstm_contig_E4_residuals_h{HORIZON}.csv"

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Output dir: {OUTPUT_DIR}")
print(f"Horizon:    {HORIZON}")
print(f"Fold:       {FOLD_NAME}")
print(f"Device:     {DEVICE}")

## Module: evaluation/splits

`split_temporal` + `winsorize_train_p99` (train-only threshold, all splits).

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame, fold: Fold = MAIN_FOLD) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]
    Fold, MAIN_FOLD, ROLLING_FOLDS, fold_by_name

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.

Rolling origin
--------------
Every published result rests on ONE test window of 22 days (February 2024), so
nothing distinguishes "the method works" from "those 22 days happened to
cooperate" — the standard objection to a forecasting evaluation, and the one the
results document still declares open.

``ROLLING_FOLDS`` answers it by re-running the whole protocol at three origins,
each with its own 22-day test window. The window **expands** rather than slides:
every fold trains from the first day of data up to its own cutoff. That is the
usual "rolling origin with recalibration" design, and it has a second payoff
here — because the folds differ in training length as well as in period, a
result that holds across all three is robust to both.

The last fold is **exactly** the main split. That is deliberate: it makes the
published result the final origin of the sequence rather than a separate
analysis, and it means only two additional folds need training.

Fold boundaries are contiguous by construction: one fold's test window becomes
the next fold's validation window. No fold ever sees its own test period during
training, which is the only property that matters.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


@dataclass(frozen=True)
class Fold:
    """One evaluation origin: three contiguous, ordered date ranges.

    Validated on construction. The invariants are not stylistic — a gap between
    ranges would silently drop days, and an overlap would train a model on its
    own test period, which is the failure this whole class exists to make
    impossible to introduce by hand.
    """

    name: str
    train_start: date
    train_end: date
    val_start: date
    val_end: date
    test_start: date
    test_end: date

    def __post_init__(self) -> None:
        for label, start, end in (
            ("train", self.train_start, self.train_end),
            ("val", self.val_start, self.val_end),
            ("test", self.test_start, self.test_end),
        ):
            if start > end:
                raise ValueError(
                    f"fold {self.name!r}: {label} range is inverted "
                    f"({start} > {end})"
                )
        day = timedelta(days=1)
        if self.val_start != self.train_end + day:
            raise ValueError(
                f"fold {self.name!r}: val must start the day after train ends; "
                f"train ends {self.train_end}, val starts {self.val_start}"
            )
        if self.test_start != self.val_end + day:
            raise ValueError(
                f"fold {self.name!r}: test must start the day after val ends; "
                f"val ends {self.val_end}, test starts {self.test_start}"
            )

    @property
    def train_days(self) -> int:
        return (self.train_end - self.train_start).days + 1

    @property
    def val_days(self) -> int:
        return (self.val_end - self.val_start).days + 1

    @property
    def test_days(self) -> int:
        return (self.test_end - self.test_start).days + 1

    def bounds(self) -> dict[str, tuple[date, date]]:
        """Split-name → (start, end), the shape the builders iterate."""
        return {
            "train": (self.train_start, self.train_end),
            "val": (self.val_start, self.val_end),
            "test": (self.test_start, self.test_end),
        }


#: The published split. Every frozen digest and every committed table is keyed
#: to it, so its dates must keep matching the module constants above.
MAIN_FOLD = Fold(
    name="main",
    train_start=SPLIT_TRAIN_START, train_end=SPLIT_TRAIN_END,
    val_start=SPLIT_VAL_START,     val_end=SPLIT_VAL_END,
    test_start=SPLIT_TEST_START,   test_end=SPLIT_TEST_END,
)

#: Three origins, oldest first. Each test window is 22 days, matching the main
#: split, so the folds differ in WHEN they are evaluated and in how much history
#: they were given — not in how much evidence each verdict rests on.
#:
#: Christmas and New Year land inside fold 1's test window and fold 2's
#: validation window. That is not a flaw to design around: if the result depends
#: on the holiday period, this is the analysis that has to reveal it.
ROLLING_FOLDS: tuple[Fold, ...] = (
    Fold(
        name="r1",
        train_start=date(2023, 10, 1), train_end=date(2023, 11, 30),
        val_start=date(2023, 12, 1),   val_end=date(2023, 12, 22),
        test_start=date(2023, 12, 23), test_end=date(2024, 1, 13),
    ),
    Fold(
        name="r2",
        train_start=date(2023, 10, 1), train_end=date(2023, 12, 22),
        val_start=date(2023, 12, 23),  val_end=date(2024, 1, 13),
        test_start=date(2024, 1, 14),  test_end=date(2024, 2, 4),
    ),
    MAIN_FOLD,
)


def fold_by_name(name: str) -> Fold:
    """Look up a fold by name, failing loudly on a typo.

    Builders take the fold as a string (CLI argument, notebook parameter), and a
    silent fallback to the main fold would produce results labelled as one origin
    and computed on another.
    """
    for fold in ROLLING_FOLDS:
        if fold.name == name:
            return fold
    known = ", ".join(fold.name for fold in ROLLING_FOLDS)
    raise KeyError(f"unknown fold {name!r}; known folds: {known}")


def split_temporal(df: pl.DataFrame, fold: Fold = MAIN_FOLD) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against ``fold``'s six
    dates. Rows outside all three ranges receive null — expected for the rolling
    folds, whose windows end before the data does, and not expected for the main
    fold (the harness raises if found there).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.
    fold:
        Evaluation origin. Defaults to :data:`MAIN_FOLD`, so every existing
        caller keeps its exact behaviour and every frozen digest stays valid.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= fold.train_start) & (day <= fold.train_end))
          .then(pl.lit("train"))
          .when((day >= fold.val_start) & (day <= fold.val_end))
          .then(pl.lit("val"))
          .when((day >= fold.test_start) & (day <= fold.test_end))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` and `rmse` in minutes.

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: data/windowing

Embedded for `compute_max_N` only. `make_window_index` is NOT used here — the retrained pipeline anchors on `make_sample_index` instead.

In [ ]:
"""Windowing module for supervised dataset construction — Fase 3 DL.

AC-WIN-1: Build window index per slot.
AC-WIN-2: Stride-parametrized index generation.
AC-WIN-3: Deterministic slot-boundary-respecting index.
AC-WIN-4: Exported constants DEFAULT_T_IN, DEFAULT_T_OUT, DEFAULT_STRIDE.
AC-WIN-5: Empty-slot guard (returns zero entries when N < T_in + T_out).
AC-WIN-6: Zero torch imports at module level.
AC-MAXN-1: compute_max_N returns train-p99 of (n_buses-1) per (empresaid, direction).
AC-MAXN-2: compute_max_N is called on train-only df; leakage is caller responsibility.

Design decisions (locked in design §2.2 and §5):
  - WindowIndexEntry: TypedDict with empresaid, direction, pair_rank, start_idx.
  - start_idx is relative to the sorted slot frame (not the full df).
  - Slot key: (empresaid, direction, pair_rank).
  - No torch imports anywhere in this module (INV-10, DL-10).
"""
from __future__ import annotations

import math
from typing import TypedDict

import polars as pl

# ---------------------------------------------------------------------------
# Constants (locked in design §5 — DL-1)
# ---------------------------------------------------------------------------

DEFAULT_T_IN: int = 12
DEFAULT_T_OUT: int = 1
DEFAULT_STRIDE: int = 1

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class WindowIndexEntry(TypedDict):
    """Single window anchor.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    pair_rank: int — positional slot index within a snapshot.
    start_idx: int — row index into the sorted slot frame where this window starts.
                     The window covers rows [start_idx, start_idx + T_in + T_out).
    """

    empresaid: int
    direction: int
    pair_rank: int
    start_idx: int


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _slot_lengths(df: pl.DataFrame) -> pl.DataFrame:
    """Return a DataFrame with (empresaid, direction, pair_rank, n_rows).

    Used by make_window_index to determine how many windows each slot produces.
    The count is over ALL rows (null delta_t_min counts — windowing does not
    drop null rows; the Dataset layer handles null masking later).
    """
    return (
        df.group_by(_SLOT_COLS)
        .agg(pl.len().alias("n_rows"))
    )


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_max_N(
    train_df: pl.DataFrame,
    *,
    quantile: float = 0.99,
) -> dict[tuple[int, int], int]:
    """Train-p99 of (n_buses - 1) per (empresaid, direction). DL-5. AC-MAXN-1..2.

    Parameters
    ----------
    train_df:
        DataFrame filtered to train rows only (caller responsibility).
        Must have columns: empresaid (Int64), direction (Int64), n_buses (Int32).
    quantile:
        Percentile for the cap (default 0.99 per DL-5).

    Returns
    -------
    dict[(empresaid, direction), int] — the maximum slot index (0-based max_N).
    Returned values are Python int (not np.int64) so they can be used as tensor
    dimensions directly.
    """
    # Compute quantile of (n_buses - 1) per (empresaid, direction).
    # We use unique snapshots: each row in the windowing context represents one
    # (empresaid, direction, snapshot) combination. n_buses is per snapshot.
    result: dict[tuple[int, int], int] = {}

    # Group by (empresaid, direction) and compute the p99 of (n_buses - 1).
    stats = (
        train_df
        .with_columns(
            (pl.col("n_buses") - 1).alias("_n_slots")
        )
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("_n_slots").quantile(quantile).alias("max_N_float")
        )
    )

    for row in stats.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        result[key] = int(math.floor(row["max_N_float"]))

    return result


def make_window_index(
    df: pl.DataFrame,
    *,
    T_in: int = DEFAULT_T_IN,
    T_out: int = DEFAULT_T_OUT,
    horizon: int | None = None,
    stride: int = DEFAULT_STRIDE,
) -> list[WindowIndexEntry]:
    """Deterministic per-slot window index. DL-1, DL-11. AC-WIN-1..5, AC-WIN-H1..H3.

    Produces a list of WindowIndexEntry dicts where each entry anchors one
    training window. Entries are sorted by (empresaid, direction, pair_rank,
    start_idx) for determinism.

    Parameters
    ----------
    df:
        headways DataFrame sorted (or sortable) by (slot_cols, t).
        Columns required: empresaid, direction, pair_rank, t.
    T_in:
        Input sequence length (number of timesteps fed to model).
    T_out:
        Prediction sequence length (number of future timesteps). Retained for
        backward compatibility. Default 1.
    horizon:
        DIRECT-horizon prediction offset. When provided, ``window_size = T_in + horizon``
        (overrides the T_out contribution). Default ``None`` falls back to T_out semantics
        so existing callers are unaffected. ``horizon=1`` produces results identical to
        ``T_out=1`` (AC-WIN-H3).
    stride:
        Step between consecutive window starts (default 1 = every timestep).

    Returns
    -------
    list[WindowIndexEntry] — may be empty if no slot has enough rows.
    """
    window_size = T_in + (horizon if horizon is not None else T_out)
    index: list[WindowIndexEntry] = []

    # Partition by slot to keep slot boundaries clean (AC-WIN-3).
    slots = df.sort(_SLOT_COLS + ["t"]).partition_by(_SLOT_COLS, maintain_order=True)

    for slot_df in slots:
        if slot_df.is_empty():
            continue

        n_rows = len(slot_df)
        if n_rows < window_size:
            # AC-WIN-5: not enough rows for even one window — skip.
            continue

        # Extract slot key from first row.
        first = slot_df.row(0, named=True)
        emp: int = int(first["empresaid"])
        direction: int = int(first["direction"])
        pr: int = int(first["pair_rank"])

        # Generate start indices with stride.
        # Number of valid windows: floor((n_rows - window_size) / stride) + 1
        n_windows = math.floor((n_rows - window_size) / stride) + 1
        for w in range(n_windows):
            start_idx = w * stride
            index.append(
                WindowIndexEntry(
                    empresaid=emp,
                    direction=direction,
                    pair_rank=pr,
                    start_idx=start_idx,
                )
            )

    return index

## Module: data/sample_index  (contracts C1 + C2)

`make_sample_index` — timestamp-anchored, contiguity-checked, one row per target.

In [ ]:
"""Canonical sample index — the shared population contract (C1 + C2).

This module exists because the project never wrote down what a sample *is*.
``windowing.make_window_index`` anchors windows on a **row index** inside a
``(empresaid, direction, pair_rank)`` slot, which produces two defects:

  C1 violation — the target of a snapshot is emitted once per anchoring slot,
  so every target is counted 2.4-5.4 times and the reported MAE is a
  fleet-density-weighted mean.

  C2 violation — consecutive row positions are not checked to be consecutive
  minutes, so the nominal horizon is a row offset, not a time offset. A window
  crossing a day boundary or a trip cut yields a "10-minute" target that is
  hours away.

``windowing.make_window_index`` is deliberately left untouched: notebooks 12/13
(and the E4 twins 18/19) must keep reproducing the frozen architecture
comparison, whose validity rests on all three architectures sharing the same
flaw. This module is the population for the *retrained* pipeline only.

Contract enforced here
----------------------
C1  A sample is ``(empresaid, direction, start_ts, horizon)``. The anchor is an
    instant, not a row position, and each one is emitted exactly once.
C2  A sample is valid only when the ``T_in + horizon`` snapshot timestamps
    starting at ``start_ts`` are strictly consecutive minutes, so the target
    lands exactly ``horizon`` minutes after the end of the input window.

``pair_rank`` survives as the position *within* the predicted vector, never as
an anchoring axis.
"""
from __future__ import annotations

from typing import TypedDict

import numpy as np
import polars as pl

# One snapshot per minute: the grid the headway series is resampled onto.
GRID_STEP_MINUTES: int = 1

# Columns that identify one snapshot series. `pair_rank` is intentionally absent.
_SERIES_COLS: list[str] = ["empresaid", "direction"]


class SampleIndexEntry(TypedDict):
    """One canonical sample.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    start_ts: the timestamp of the FIRST input snapshot.
    target_ts: the timestamp of the predicted snapshot. Always exactly
               ``horizon`` minutes after the last input snapshot.
    horizon: int — prediction offset in minutes (now genuinely minutes).
    """

    empresaid: int
    direction: int
    start_ts: object
    target_ts: object
    horizon: int


def _contiguous_run_mask(ts: np.ndarray, span: int) -> np.ndarray:
    """Mask of window starts whose ``span`` timestamps are consecutive minutes.

    Parameters
    ----------
    ts:
        Sorted, unique timestamps for one series, as numpy datetime64.
    span:
        Number of consecutive timestamps a valid window must cover
        (``T_in + horizon``).

    Returns
    -------
    Boolean array of length ``len(ts)``. ``mask[i]`` is True when
    ``ts[i:i+span]`` are consecutive minutes. Positions with fewer than
    ``span`` timestamps remaining are False.
    """
    n = ts.size
    mask = np.zeros(n, dtype=bool)
    if n < span:
        return mask

    step = np.timedelta64(GRID_STEP_MINUTES, "m")
    # gap[i] is True when ts[i+1] follows ts[i] by exactly one grid step.
    gap_ok = np.diff(ts) == step

    # A window at i needs the span-1 gaps starting at i to all be contiguous.
    # Cumulative-sum trick: count of good gaps in [i, i+span-2] must equal span-1.
    need = span - 1
    if need == 0:
        mask[:] = True
        return mask

    cum = np.concatenate(([0], np.cumsum(gap_ok)))
    starts = np.arange(n - span + 1)
    good = cum[starts + need] - cum[starts]
    mask[: n - span + 1] = good == need
    return mask


def make_sample_index(
    df: pl.DataFrame,
    *,
    horizon: int,
    T_in: int,
) -> pl.DataFrame:
    """Canonical sample index for one frame. Enforces C1 and C2.

    Parameters
    ----------
    df:
        Headway frame. Required columns: ``empresaid``, ``direction``, ``t``.
        ``pair_rank`` may be present; it is ignored for anchoring.
    horizon:
        Prediction offset in minutes.
    T_in:
        Input window length in snapshots.

    Returns
    -------
    DataFrame with one row per canonical sample, columns
    ``empresaid, direction, start_ts, target_ts, horizon``, sorted
    deterministically. Empty (with the right schema) when no window qualifies.
    """
    if horizon < 1:
        raise ValueError(f"horizon must be >= 1, got {horizon}")
    if T_in < 1:
        raise ValueError(f"T_in must be >= 1, got {T_in}")

    span = T_in + horizon
    rows: list[pl.DataFrame] = []

    # One series per (empresaid, direction) — NOT per pair_rank. That collapse is
    # the whole point of C1: the target is the snapshot, not the slot.
    series = (
        df.select(_SERIES_COLS + ["t"])
        .unique()
        .sort(_SERIES_COLS + ["t"])
        .partition_by(_SERIES_COLS, maintain_order=True)
    )

    for s in series:
        if s.is_empty():
            continue
        ts = s.get_column("t").to_numpy()
        mask = _contiguous_run_mask(ts, span)
        if not mask.any():
            continue

        first = s.row(0, named=True)
        starts = ts[mask]
        # The target sits `horizon` minutes after the LAST input snapshot, which
        # is start + (T_in - 1) steps. Contiguity is already guaranteed by mask.
        targets = starts + np.timedelta64((T_in - 1 + horizon) * GRID_STEP_MINUTES, "m")

        rows.append(
            pl.DataFrame(
                {
                    "empresaid": np.full(starts.size, int(first["empresaid"]), dtype=np.int64),
                    "direction": np.full(starts.size, int(first["direction"]), dtype=np.int64),
                    "start_ts": starts,
                    "target_ts": targets,
                    "horizon": np.full(starts.size, int(horizon), dtype=np.int64),
                }
            )
        )

    if not rows:
        return pl.DataFrame(
            schema={
                "empresaid": pl.Int64,
                "direction": pl.Int64,
                "start_ts": df.schema["t"],
                "target_ts": df.schema["t"],
                "horizon": pl.Int64,
            }
        )

    return pl.concat(rows).sort(["empresaid", "direction", "start_ts"])


def effective_horizon_minutes(
    index: pl.DataFrame,
    *,
    T_in: int,
) -> pl.Series:
    """Realized gap in minutes between end-of-window and target, per sample.

    Under C2 this is constant and equal to ``horizon`` for every row. It is
    exposed so a test can assert that rather than trust the construction.
    """
    window_end = pl.col("start_ts") + pl.duration(minutes=(T_in - 1) * GRID_STEP_MINUTES)
    return (
        index.select(
            ((pl.col("target_ts") - window_end).dt.total_minutes()).alias("eff")
        )
        .get_column("eff")
    )

## Module: data/normalization

z-score stats from TRAIN ONLY; applied to every split.

In [ ]:
"""Normalization module for supervised dataset construction — Fase 3 DL.

AC-NORM-1: compute_normalization_stats uses TRAIN ROWS ONLY.
AC-NORM-2: apply_zscore = (x - mean) / (std + Z_EPS) per (empresaid, direction).
AC-NORM-3: null delta_t_min passes through as null in the output column.
AC-NORM-4: no clipping — values with |z| > 5 are passed through unmodified (DL-8).
AC-NORM-5: zero torch imports at module level (INV-10).
AC-LEAK-1: leakage guard — caller must pass train_df only; this module does not filter.

Pre-condition: input df must already be winsorized via winsorize_train_p99 (INV-6).
Design decisions locked in design §2.3 and §5.
"""
from __future__ import annotations

from dataclasses import dataclass

import polars as pl

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

Z_EPS: float = 1e-8  # numerical safety: (x - mean) / (std + Z_EPS)


# ---------------------------------------------------------------------------
# Data types
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class NormalizationStats:
    """Per-(empresaid, direction) z-score parameters. Pure data, no torch.

    Attributes
    ----------
    means:
        Mean of delta_t_min per (empresaid, direction) computed from train rows only.
    stds:
        Standard deviation of delta_t_min per (empresaid, direction) from train only.
    """

    means: dict[tuple[int, int], float]
    stds: dict[tuple[int, int], float]


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _lookup_expr(
    stats: NormalizationStats,
    kind: str,
) -> pl.Expr:
    """Build a polars conditional expression mapping (empresaid, direction) → mean|std.

    Instead of a Python row-by-row loop, we build a pl.when/then chain over all
    known (empresa, direction) keys. Unknown keys return 0.0 (should not happen
    in a correctly filtered input; callers are expected to pass frames that
    only contain corridors present in train).

    Parameters
    ----------
    stats:
        NormalizationStats holding all known keys.
    kind:
        Either "mean" or "std".
    """
    lookup = stats.means if kind == "mean" else stats.stds

    if not lookup:
        return pl.lit(0.0)

    items = list(lookup.items())
    (emp0, dir0), val0 = items[0]
    expr = pl.when(
        (pl.col("empresaid") == emp0) & (pl.col("direction") == dir0)
    ).then(pl.lit(val0))

    for (emp, direction), val in items[1:]:
        expr = expr.when(
            (pl.col("empresaid") == emp) & (pl.col("direction") == direction)
        ).then(pl.lit(val))

    return expr.otherwise(pl.lit(0.0))


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_normalization_stats(
    train_df: pl.DataFrame,
) -> NormalizationStats:
    """Mean/std of delta_t_min per (empresaid, direction) from TRAIN ROWS ONLY.

    AC-NORM-1: caller must pass a train-only DataFrame (no leakage protection
    inside this function — leakage guard is the caller's responsibility per INV-2).

    Null delta_t_min rows are excluded from the computation (standard mean/std
    ignores nulls in polars by default).

    Parameters
    ----------
    train_df:
        DataFrame with train rows only. Required columns: empresaid (Int64),
        direction (Int64), delta_t_min (Float64 nullable).

    Returns
    -------
    NormalizationStats with means and stds dicts keyed by (empresaid, direction).
    """
    agg = (
        train_df
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("delta_t_min").mean().alias("mean"),
            pl.col("delta_t_min").std().alias("std"),
        )
    )

    means: dict[tuple[int, int], float] = {}
    stds: dict[tuple[int, int], float] = {}

    for row in agg.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        means[key] = float(row["mean"]) if row["mean"] is not None else 0.0
        stds[key] = float(row["std"]) if row["std"] is not None else 0.0

    return NormalizationStats(means=means, stds=stds)


def apply_zscore(
    df: pl.DataFrame,
    stats: NormalizationStats,
    *,
    out_col: str = "delta_t_min_z",
) -> pl.DataFrame:
    """Add z-scored column: (delta_t_min - mean) / (std + Z_EPS) per (empresa, direction).

    AC-NORM-2: formula is (x - mean) / (std + Z_EPS).
    AC-NORM-3: null delta_t_min rows produce null in out_col (no imputation).
    AC-NORM-4 + DL-8: no clipping — values with |z| > 5 pass through unchanged.

    Parameters
    ----------
    df:
        DataFrame to z-score. May be train, val, or test split. Required columns:
        empresaid, direction, delta_t_min.
    stats:
        NormalizationStats from compute_normalization_stats (train only).
    out_col:
        Name for the output z-scored column (default: delta_t_min_z).

    Returns
    -------
    pl.DataFrame — input frame with out_col (Float64 nullable) added.
    """
    mean_expr = _lookup_expr(stats, "mean")
    std_expr = _lookup_expr(stats, "std")

    return df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
        .then(None)
        .otherwise(
            (pl.col("delta_t_min") - mean_expr) / (std_expr + Z_EPS)
        )
        .alias(out_col)
        .cast(pl.Float64)
    )

## Module: data/context_features

`encode_context` still emits the 5-column set; the retrained pipeline consumes only the 4 causal ones (`CAUSAL_CONTEXT_FEATURE_NAMES`).

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    # The frozen 02-eda-corridors CSV names its date column `day`; older
    # fixtures use `date`. Accept either, preferring `date` when both exist.
    date_col = next((c for c in ("date", "day") if c in df.columns), None)
    if date_col is None:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' or 'day' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df[date_col].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: data/contiguous_dataset  (first torch import)

`materialize_arrays` — the path this notebook takes, covered by `tests/data/test_contiguous_dataset.py` rather than reimplemented in a cell.

In [ ]:
"""Dataset backed by the canonical sample index — the retrained pipeline's loader.

``HeadwayDataset`` (``src/data/dataset.py``) anchors every window inside a
``(empresaid, direction, pair_rank)`` slot and slices it by row position. Two
consequences, both audit findings:

  * the same snapshot target is emitted once per anchoring slot, so the reported
    MAE is weighted by fleet density (#13);
  * row positions are not checked to be consecutive minutes, so the nominal
    horizon is a row offset rather than a time offset (§3).

This loader consumes ``sample_index.make_sample_index`` instead. Anchors are
timestamps, each target appears exactly once, and contiguity is guaranteed by
construction — so the window timestamps can be derived arithmetically rather
than read off a slot frame.

``HeadwayDataset`` is left in place and untouched: notebooks 12/13/18/19 must
keep reproducing the frozen architecture comparison, whose validity rests on all
three architectures sharing the same flaw.

Context features
----------------
``CAUSAL_CONTEXT_FEATURE_NAMES`` drops ``atypical_flag`` from the five-column
set. The flag is a whole-day aggregate whose threshold was fitted over all 152
days including test, so classifying a day requires that day's full record count
— information unavailable at 08:00 on the day being predicted. That is leakage
by design, not by parametrization, so the feature is removed rather than
recalibrated (see ``docs/plan-reentrenamiento.md`` §2, C3).
"""
from __future__ import annotations

import numpy as np
import polars as pl
import torch
from torch.utils.data import Dataset

# Cyclical time only. `atypical_flag` is deliberately absent — see module docstring.
CAUSAL_CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
)

_SERIES_COLS: list[str] = ["empresaid", "direction"]


class _SeriesGrid:
    """Dense (n_timesteps, max_N) view of one ``(empresaid, direction)`` series.

    Built once per series and reused by every sample that anchors into it. This
    replaces the per-timestep ``filter`` the legacy loader performed inside
    ``__getitem__``, which cost one full scan per timestep of every window.

    The grid holds only distinct snapshots, so its footprint is
    ``n_timesteps x max_N`` — independent of how many windows anchor into it.
    """

    __slots__ = ("ts_index", "ts_values", "values", "masks", "context")

    def __init__(
        self,
        frame: pl.DataFrame,
        *,
        max_N: int,
        value_col: str,
        context_cols: list[str],
    ) -> None:
        timestamps = (
            frame.select("t").unique().sort("t").get_column("t").to_numpy()
        )
        self.ts_index: dict[np.datetime64, int] = {
            ts: i for i, ts in enumerate(timestamps)
        }
        # Positional view of the same axis, so a target row can be verified
        # against the timestamp the sample index recorded.
        self.ts_values = timestamps
        n = timestamps.size

        self.values = np.zeros((n, max_N), dtype=np.float32)
        self.masks = np.zeros((n, max_N), dtype=bool)
        self.context = np.zeros((n, len(context_cols)), dtype=np.float32)

        rows = frame.select(["t", "pair_rank", value_col] + context_cols)
        ts_col = rows.get_column("t").to_numpy()
        pr_col = rows.get_column("pair_rank").to_numpy()
        val_col = np.asarray(rows.get_column(value_col).to_numpy(), dtype=np.float64)

        row_of = np.array([self.ts_index[ts] for ts in ts_col], dtype=np.int64)

        # Values and masks: only in-range pair_ranks with an observed value.
        # Out-of-range slots are truncated at max_N (AC-MAXN-2); missing values
        # keep value 0.0 and mask False, the legacy null convention (AC-MASK-3).
        #
        # Missingness MUST be tested with np.isnan, not polars' is_null. Once a
        # column is converted to numpy its nulls become NaN and the null flag is
        # gone, so `pl.Series(arr).is_null()` answers False for every one of them
        # — polars treats NaN and null as different things. Getting this wrong
        # writes NaN into the tensor with mask=True, which makes the loss NaN,
        # leaves every epoch un-improved, and yields an empty state_dict.
        in_range = (pr_col >= 0) & (pr_col < max_N)
        present = in_range & ~np.isnan(val_col)
        self.values[row_of[present], pr_col[present]] = val_col[present].astype(np.float32)
        self.masks[row_of[present], pr_col[present]] = True

        # Context is per snapshot, identical across pair_ranks — last write wins.
        for c_idx, name in enumerate(context_cols):
            col = rows.get_column(name).fill_null(0.0).to_numpy()
            self.context[row_of, c_idx] = col.astype(np.float32)


def materialize_arrays(
    df: pl.DataFrame,
    sample_index: pl.DataFrame,
    *,
    max_N: int,
    T_in: int,
    horizon: int,
    value_col: str = "delta_t_min_z",
    context_cols: tuple[str, ...] = CAUSAL_CONTEXT_FEATURE_NAMES,
) -> dict[str, np.ndarray]:
    """Dense arrays for every sample, in sample-index row order.

    The Kaggle notebooks materialize the whole split up front rather than
    streaming through a DataLoader, so this is the path they take. It lives here
    — beside the Dataset and covered by the same tests — instead of being
    reimplemented inside a notebook cell, because an untested copy inside a
    generated artifact is how the pipeline drifted from its contracts before.

    Returns
    -------
    dict with ``input`` (n, T_in, max_N), ``target`` (n, 1, max_N),
    ``input_mask``, ``target_mask``, ``context`` (n, T_in, n_ctx) — the same key
    names ``collate_fn`` and ``train.py`` already consume.

    Raises
    ------
    ValueError
        When a sample's target does not land on the timestamp the index
        declared, i.e. contract C2 is violated for that row.
    """
    if "atypical_flag" in context_cols:
        raise ValueError(
            "atypical_flag is a leaking feature and must not be materialized "
            "in the retrained pipeline (plan-reentrenamiento.md C3)"
        )

    n = sample_index.height
    n_ctx = len(context_cols)
    out = {
        "input": np.zeros((n, T_in, max_N), dtype=np.float32),
        "target": np.zeros((n, 1, max_N), dtype=np.float32),
        "input_mask": np.zeros((n, T_in, max_N), dtype=bool),
        "target_mask": np.zeros((n, 1, max_N), dtype=bool),
        "context": np.zeros((n, T_in, n_ctx), dtype=np.float32),
    }
    if n == 0:
        return out

    emp_col = sample_index.get_column("empresaid").to_numpy()
    dir_col = sample_index.get_column("direction").to_numpy()
    start_col = sample_index.get_column("start_ts").to_numpy()
    target_col = sample_index.get_column("target_ts").to_numpy()

    order = np.arange(n)
    for (empresaid, direction) in sorted({(int(e), int(d)) for e, d in zip(emp_col, dir_col)}):
        rows = order[(emp_col == empresaid) & (dir_col == direction)]
        if rows.size == 0:
            continue
        series = df.filter(
            (pl.col("empresaid") == empresaid) & (pl.col("direction") == direction)
        )
        grid = _SeriesGrid(
            series,
            max_N=max_N,
            value_col=value_col,
            context_cols=list(context_cols),
        )
        starts = np.array([grid.ts_index[ts] for ts in start_col[rows]], dtype=np.int64)
        target_rows = starts + (T_in - 1 + horizon)

        if target_rows.max() >= grid.ts_values.size:
            raise ValueError(
                f"target row out of range for series ({empresaid}, {direction})"
            )
        declared = target_col[rows]
        if not np.array_equal(grid.ts_values[target_rows], declared):
            bad = int(np.argmax(grid.ts_values[target_rows] != declared))
            raise ValueError(
                f"contiguity violated for ({empresaid}, {direction}) "
                f"start={start_col[rows][bad]}: grid holds "
                f"{grid.ts_values[target_rows][bad]}, index declared {declared[bad]}"
            )

        # Window rows are start..start+T_in-1, contiguous by C2.
        window_rows = starts[:, None] + np.arange(T_in)[None, :]
        out["input"][rows] = grid.values[window_rows]
        out["input_mask"][rows] = grid.masks[window_rows]
        out["context"][rows] = grid.context[window_rows]
        out["target"][rows, 0] = grid.values[target_rows]
        out["target_mask"][rows, 0] = grid.masks[target_rows]

    return out


class ContiguousHeadwayDataset(Dataset):
    """Sample-index-backed dataset. One item per canonical sample.

    Tensor contract matches ``HeadwayDataset`` so ``train.py`` and ``collate_fn``
    need no changes:
        input       : (T_in, max_N)   float32
        target      : (1, max_N)      float32
        input_mask  : (T_in, max_N)   bool
        target_mask : (1, max_N)      bool
        context     : (T_in, n_ctx)   float32   — 4 columns, no atypical flag

    Mask polarity: True = VALID (INV-5).
    """

    def __init__(
        self,
        df: pl.DataFrame,
        sample_index: pl.DataFrame,
        *,
        max_N_by_direction: dict[tuple[int, int], int],
        T_in: int,
        horizon: int,
        value_col: str = "delta_t_min_z",
        context_cols: tuple[str, ...] = CAUSAL_CONTEXT_FEATURE_NAMES,
    ) -> None:
        if horizon < 1:
            raise ValueError(f"horizon must be >= 1, got {horizon}")
        missing = [c for c in context_cols if c not in df.columns]
        if missing:
            raise ValueError(f"df is missing context columns: {missing}")
        if "atypical_flag" in context_cols:
            raise ValueError(
                "atypical_flag is a leaking feature and must not be a context "
                "column in the retrained pipeline (plan-reentrenamiento.md C3)"
            )

        self._df = df
        self._index = sample_index
        self._max_N_by_direction = max_N_by_direction
        self._T_in = T_in
        self._horizon = horizon
        self._value_col = value_col
        self._context_cols = list(context_cols)

        # Materialized lazily per series — never per window (INV-7).
        self._grids: dict[tuple[int, int], _SeriesGrid] = {}

        # Column-oriented access beats row(named=True) inside __getitem__.
        self._emp = sample_index.get_column("empresaid").to_numpy()
        self._dir = sample_index.get_column("direction").to_numpy()
        self._start = sample_index.get_column("start_ts").to_numpy()
        self._target = sample_index.get_column("target_ts").to_numpy()

    def __len__(self) -> int:
        return self._index.height

    def _grid(self, empresaid: int, direction: int) -> _SeriesGrid:
        key = (empresaid, direction)
        if key not in self._grids:
            frame = self._df.filter(
                (pl.col("empresaid") == empresaid)
                & (pl.col("direction") == direction)
            )
            self._grids[key] = _SeriesGrid(
                frame,
                max_N=self._max_N_by_direction[key],
                value_col=self._value_col,
                context_cols=self._context_cols,
            )
        return self._grids[key]

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        empresaid = int(self._emp[idx])
        direction = int(self._dir[idx])
        start_ts = self._start[idx]
        target_ts = self._target[idx]

        grid = self._grid(empresaid, direction)
        i = grid.ts_index[start_ts]
        target_row = i + self._T_in - 1 + self._horizon

        # Defense in depth: C2 guarantees the run is contiguous, so the target
        # must land on exactly the timestamp the index recorded. If it does not,
        # the index and the frame disagree and the sample is silently wrong —
        # precisely the failure mode this pipeline exists to prevent.
        if target_row >= grid.ts_values.size:
            raise IndexError(
                f"target row {target_row} out of range for series "
                f"({empresaid}, {direction}) with {grid.ts_values.size} snapshots"
            )
        if grid.ts_values[target_row] != target_ts:
            raise ValueError(
                f"contiguity violated for ({empresaid}, {direction}) "
                f"start={start_ts}: target row holds "
                f"{grid.ts_values[target_row]}, index declared {target_ts}"
            )

        values = torch.from_numpy(grid.values[i : i + self._T_in].copy())
        masks = torch.from_numpy(grid.masks[i : i + self._T_in].copy())
        context = torch.from_numpy(grid.context[i : i + self._T_in].copy())
        target = torch.from_numpy(grid.values[target_row : target_row + 1].copy())
        target_mask = torch.from_numpy(grid.masks[target_row : target_row + 1].copy())

        return {
            "input": values,
            "target": target,
            "input_mask": masks,
            "target_mask": target_mask,
            "context": context,
        }

## Module: evaluation/residual_export

`build_keyed_residuals` + `assert_key_is_unique` — the full-key contract.

In [ ]:
"""Canonical per-sample residual export — the full-key contract.

Every question this project could not answer from disk traces to an export that
threw the key away:

  * ``harness.py`` exported XGBoost residuals keyed on ``t`` alone, which is not
    unique (~4.49 rows per ``(t, direction)``). Repairing the DL-vs-XGBoost
    comparison therefore needed a whole new Kaggle kernel, ``20-xgb-paired-export``
    (audit §2.1).
  * The DL residual CSVs carry only
    ``corridor, direction, horizon, y_true, y_pred_dl, y_pred_persist`` — no
    ``t``, no ``pair_rank`` — which is why clustering by service day (#6) and a
    per-position error profile (#5) cannot be computed locally at all.

Both are the same mistake: a lossy export turns every new question into another
GPU run. This module fixes it once, for every model family.

The key
-------
``(corridor, direction, horizon, split, start_ts, target_ts, pair_rank)`` is
unique by construction: the sample index guarantees one row per
``(empresaid, direction, start_ts, horizon)`` (contract C1) and ``pair_rank``
indexes position within that sample's predicted vector.

Reconstruction
--------------
Model outputs arrive as dense ``(n_samples, max_N)`` arrays whose row order is
the sample index's row order and whose column order is ``pair_rank``. The key is
therefore recoverable exactly: repeat each index row ``max_N`` times, tile
``pair_rank`` across it, then drop masked-out cells. No join, no ambiguity.
"""
from __future__ import annotations

import numpy as np
import polars as pl

# Canonical column order. Key first, then values — so a `head` on the CSV shows
# what identifies a row before what it measured.
RESIDUAL_KEY_COLUMNS: list[str] = [
    "corridor",
    "direction",
    "horizon",
    "split",
    "start_ts",
    "target_ts",
    "pair_rank",
]

RESIDUAL_VALUE_COLUMNS: list[str] = [
    "y_true",
    "y_pred_model",
    "y_pred_persist",
]

RESIDUAL_COLUMNS: list[str] = RESIDUAL_KEY_COLUMNS + RESIDUAL_VALUE_COLUMNS


def direction_label(direction_val: int) -> str:
    """Signed direction label ("-1" / "+1").

    Kept identical to ``baselines.harness._direction_label`` and the legacy DL
    exports so the new residuals stay readable by the existing analysis layer.
    """
    return f"+{direction_val}" if direction_val > 0 else str(direction_val)


def build_keyed_residuals(
    sample_index: pl.DataFrame,
    *,
    corridor: str,
    split: str,
    y_true: np.ndarray,
    y_pred_model: np.ndarray,
    y_pred_persist: np.ndarray,
    target_mask: np.ndarray,
    persist_mask: np.ndarray,
) -> pl.DataFrame:
    """Per-sample paired residuals carrying the full key.

    Parameters
    ----------
    sample_index:
        The frame from ``make_sample_index``, in the order the model consumed
        it. Row ``i`` of every array below corresponds to row ``i`` here.
    corridor:
        Corridor label ("E2", "E59", "E4").
    split:
        Split label ("train", "val", "test").
    y_true, y_pred_model, y_pred_persist:
        Dense ``(n_samples, max_N)`` arrays in original (un-z-scored) units.
    target_mask, persist_mask:
        Dense ``(n_samples, max_N)`` boolean arrays, True = VALID (INV-5).
        A cell is exported only where BOTH are valid — the paired set the
        significance tests require.

    Returns
    -------
    DataFrame with ``RESIDUAL_COLUMNS``, sorted deterministically.

    Raises
    ------
    ValueError
        If any array's shape disagrees with the index height, which would mean
        the export is silently misaligned with the population it claims.
    """
    n = sample_index.height
    arrays = {
        "y_true": y_true,
        "y_pred_model": y_pred_model,
        "y_pred_persist": y_pred_persist,
        "target_mask": target_mask,
        "persist_mask": persist_mask,
    }
    shapes = {name: a.shape for name, a in arrays.items()}
    if len({s for s in shapes.values()}) != 1:
        raise ValueError(f"arrays disagree in shape: {shapes}")
    n_rows, max_N = y_true.shape
    if n_rows != n:
        raise ValueError(
            f"array rows ({n_rows}) != sample index height ({n}); the export "
            "is misaligned with its population"
        )

    keep = target_mask & persist_mask
    if not keep.any():
        return pl.DataFrame(schema={c: pl.Utf8 for c in RESIDUAL_COLUMNS})

    # Row i of the flattened arrays maps to index row i // max_N, pair_rank i % max_N.
    rows, cols = np.nonzero(keep)

    directions = sample_index.get_column("direction").to_numpy()[rows]
    starts = sample_index.get_column("start_ts").to_numpy()[rows]
    targets = sample_index.get_column("target_ts").to_numpy()[rows]
    horizons = sample_index.get_column("horizon").to_numpy()[rows]

    frame = pl.DataFrame(
        {
            "corridor": np.full(rows.size, corridor),
            "direction": [direction_label(int(d)) for d in directions],
            "horizon": horizons.astype(np.int64),
            "split": np.full(rows.size, split),
            "start_ts": starts,
            "target_ts": targets,
            "pair_rank": cols.astype(np.int64),
            "y_true": y_true[rows, cols].astype(np.float64),
            "y_pred_model": y_pred_model[rows, cols].astype(np.float64),
            "y_pred_persist": y_pred_persist[rows, cols].astype(np.float64),
        }
    )

    return frame.select(RESIDUAL_COLUMNS).sort(
        ["corridor", "direction", "horizon", "start_ts", "pair_rank"]
    )


def assert_key_is_unique(residuals: pl.DataFrame) -> None:
    """Fail closed when the exported key does not identify a row.

    This is the check whose absence cost a Kaggle kernel: ``harness.py``'s
    docstring declared ``t`` a join key and nothing verified it.
    """
    keys = residuals.select(RESIDUAL_KEY_COLUMNS)
    if keys.height != keys.unique().height:
        dupes = (
            keys.group_by(RESIDUAL_KEY_COLUMNS)
            .len()
            .filter(pl.col("len") > 1)
            .sort("len", descending=True)
        )
        raise ValueError(
            f"residual key is not unique: {dupes.height} duplicated keys, "
            f"worst multiplicity {dupes.get_column('len').max()}"
        )

## Module: models/lstm

`HeadwayLSTM` + `masked_mse_loss`.

In [ ]:
"""HeadwayLSTM — flat LSTM encoder for multi-bus headway forecasting.

Architecture decisions (from design):
  AD-1: Input is a flat concatenation of headway vector (max_N) and context (5)
        at each timestep. The caller performs the concatenation before calling
        forward. input_size = max_N + context_size.
  AD-2: Only the last hidden state h[-1] is used → Linear head → (B, output_size).
        seq2one: T_out=1 throughout Fase 5.
  AD-3: Mask is applied ONLY at the loss level. The model always produces
        output for all positions regardless of which slots are valid.

Invariants:
  INV-NO-COMPILE: torch.compile() is NOT used (Kaggle CUDA compatibility).
  INV-MASK: masked_mse_loss gates gradient on mask; model is mask-agnostic.

ACs covered: AC-MODEL-1..5, AC-LOSS-1..5.
"""
from __future__ import annotations

import torch
import torch.nn as nn


class HeadwayLSTM(nn.Module):
    """Flat LSTM encoder for multi-bus headway forecasting.

    Parameters
    ----------
    input_size:
        Size of the LSTM input at each timestep.
        Must equal max_N + context_size at the call site (AD-1, INV-INPUT-SIZE).
    hidden_size:
        Number of LSTM hidden units.
    output_size:
        Number of output positions (= max_N). The linear head maps
        hidden_size → output_size.
    num_layers:
        Number of stacked LSTM layers (default 1).
    dropout:
        Dropout probability applied between LSTM layers (default 0.0).
        Ignored when num_layers == 1 (PyTorch behaviour).
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Parameters
        ----------
        x:
            (B, T_in, input_size) — concatenated headway + context tensor.
            Caller is responsible for torch.cat([input, context], dim=-1).

        Returns
        -------
        (B, output_size) — predicted next headway vector (z-scored).
        Uses only the last hidden state (AD-2).
        """
        # lstm_out: (B, T_in, hidden_size)
        # h_n:      (num_layers, B, hidden_size)
        _lstm_out, (h_n, _c_n) = self.lstm(x)

        # Take the hidden state from the topmost layer at the last timestep.
        # h_n[-1]: (B, hidden_size)
        last_hidden = h_n[-1]

        # Project to output space: (B, output_size)
        return self.head(last_hidden)


def masked_mse_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """Mean squared error computed only over valid (mask == True) positions.

    Parameters
    ----------
    pred:
        (B, max_N) float32 — model predictions (z-scored).
    target:
        (B, max_N) float32 — ground-truth values (z-scored).
    mask:
        (B, max_N) bool — True = VALID position; False = absent/padded slot.

    Returns
    -------
    Scalar 0-dim tensor. If no position is True, returns 0.0 (clamp(min=1)
    prevents zero-division, per AD-3 and INV-MASK).

    Formula: ((pred - target)^2 * mask.float()).sum() / mask.float().sum().clamp(min=1)
    """
    mask_f = mask.float()
    squared_error = (pred - target) ** 2
    # Sum only over valid positions, then normalize by count (clamped to 1).
    return (squared_error * mask_f).sum() / mask_f.sum().clamp(min=1)

## Module: train

`TrainConfig`, `set_seed`, `train_model`, `grid_search`, `denormalize_predictions`.

In [ ]:
"""Training primitives for the LSTM headway-forecasting baseline.

Architecture decisions applied (from design doc):
  AD-1: Caller concatenates input + context before model forward.
        train_one_epoch / evaluate_epoch perform: x = cat([batch["input"], batch["context"]], dim=-1).
  AD-4: Duck-type dispatch via model.spatial attribute (Fase 6a/6b).
        If hasattr(model, 'spatial') and model.spatial, call model(inp, ctx, input_mask).
        Otherwise call model(cat([inp, ctx])) as before — HeadwayLSTM path unchanged.
  AD-5: TrainConfig gets optional conv_channels: int | None = None (Fase 6a).
  AD-6: Denormalization — pred_minutes = pred_z * (std + 1e-8) + mean.
  AD-7: Reproducibility — torch + cuda + numpy manual seeds.
  AD-9: Early stopping — monitor val masked MSE, in-memory best-state copy.
  Fase 6b AD-6: TrainConfig gets optional nhead: int | None = None,
                d_model: int | None = None for SpatialTransformer dispatch.
  Fase 6b AD-7: TRANSFORMER_GRID — 32 configs (obs #417 canonical grid):
                nhead{1,2} × d_model{16,32} × hidden{32,64} × dropout{0.0,0.2}
                × lr{1e-3,5e-4}, num_layers=1 fixed. All (nhead,d_model) combos
                satisfy d_model % nhead == 0 → 2^5 = 32 configs.
  INV-NO-COMPILE: torch.compile() is NOT used (Kaggle CUDA compatibility).
  INV-NO-TB: No TensorBoard, MLflow, or Optuna imports.

ACs covered (Wave 2):
  AC-TRAIN-1: train_one_epoch returns float.
  AC-TRAIN-2: Weights change after train_one_epoch.
  AC-TRAIN-3: evaluate_epoch returns float.
  AC-TRAIN-4: No gradients accumulated during evaluate_epoch.
  AC-TRAIN-5: EarlyStopping fires after patience non-improving epochs.
  AC-TRAIN-6: EarlyStopping.best_state_dict is a deep copy of model state.

ACs covered (Wave 3):
  AC-TRAIN-7: save_checkpoint / load_checkpoint round-trip produces identical predictions.
  AC-TRAIN-8: train_model honours early stopping; restores best weights on return.
  AC-GRID-1..5: GRID has 24 entries; grid_search returns sorted list of TrainResult.
  AC-EVAL-1: denormalize_predictions converts z-scored output back to minutes.

ACs covered (Fase 6a Wave 2):
  SPATIAL-DISPATCH: train_one_epoch / evaluate_epoch dispatch on model.spatial.
  SPATIAL-GRID: SPATIAL_GRID has 48 configs (conv_channels×hidden×layers×dropout×lr).
  SPATIAL-COMPAT: HeadwayLSTM path unchanged; all Fase 5 tests still pass.

ACs covered (Fase 6b Wave 2):
  TRANSFORMER-DISPATCH: grid_search dispatches nhead-first → SpatialTransformer.
  TRANSFORMER-GRID: TRANSFORMER_GRID has 32 configs (obs #417).
  TRANSFORMER-COMPAT: conv_channels path and HeadwayLSTM path unchanged.
"""
from __future__ import annotations

import copy
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn



# ---------------------------------------------------------------------------
# Dataclasses
# ---------------------------------------------------------------------------

@dataclass
class TrainConfig:
    """Hyperparameters for one LSTM training run.

    Required:
        hidden_size: Number of LSTM hidden units.
        num_layers:  Number of stacked LSTM layers.
        dropout:     Dropout probability applied between LSTM layers.
        lr:          Adam learning rate.

    Optional (with defaults):
        batch_size:  DataLoader batch size (default 32).
        max_epochs:  Hard ceiling on training epochs (default 50).
        patience:    Early-stopping patience in epochs (default 10).
        seed:        Global random seed (default 42).
    """

    hidden_size: int
    num_layers: int
    dropout: float
    lr: float
    batch_size: int = 32
    max_epochs: int = 50
    patience: int = 10
    seed: int = 42
    # Fase 6a — optional spatial conv channels.  None → HeadwayLSTM path (AD-5).
    conv_channels: int | None = None
    # Fase 6b — optional transformer attention parameters (AD-6).
    # nhead=None, d_model=None → no SpatialTransformer (backward compat).
    # nhead and conv_channels are mutually exclusive per config.
    nhead: int | None = None
    d_model: int | None = None


@dataclass
class TrainResult:
    """Result of one full training run (returned by train_model / grid_search).

    Attributes
    ----------
    best_val_loss:
        Lowest validation masked MSE achieved during training.
    best_epoch:
        Zero-indexed epoch at which best_val_loss was recorded.
    epochs_trained:
        Total number of epochs actually executed (including the stopping epoch).
    train_losses:
        Chronological list of per-epoch training losses.
    val_losses:
        Chronological list of per-epoch validation losses.
    state_dict:
        Model weights (deep copy) from the best epoch (AD-9).
    config:
        The TrainConfig that produced this result.
    """

    best_val_loss: float
    best_epoch: int
    epochs_trained: int = 0
    train_losses: list[float] = field(default_factory=list)
    val_losses: list[float] = field(default_factory=list)
    state_dict: dict[str, Any] = field(default_factory=dict)
    config: TrainConfig = field(default_factory=lambda: TrainConfig(64, 1, 0.0, 1e-3))


# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------

def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility (AD-7).

    Seeds:
    - torch (CPU)
    - torch.cuda (all GPU devices)
    - numpy

    Note: full determinism across CUDA hardware is NOT guaranteed — this seeds
    the major sources of randomness for same-machine reproducibility.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


# ---------------------------------------------------------------------------
# Per-epoch training and evaluation
# ---------------------------------------------------------------------------

def train_one_epoch(
    model: nn.Module,
    loader: Any,  # DataLoader or list of dict batches
    optimizer: torch.optim.Optimizer,
    device: str | torch.device = "cpu",
) -> float:
    """Train for one epoch.

    Iterates the loader; for each batch:
    1. Concatenate batch["input"] and batch["context"] along dim=-1 (AD-1).
    2. Forward pass through model → pred: (B, max_N).
    3. Squeeze target and mask from (B, 1, max_N) to (B, max_N).
    4. Compute masked_mse_loss(pred, target, mask).
    5. Backpropagate and step optimizer.

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to train.
    loader:
        Iterable yielding dicts with keys: input, context, target, target_mask.
    optimizer:
        Optimizer (e.g., Adam).
    device:
        Device string or torch.device. Tensors are moved to this device.

    Returns
    -------
    Mean masked MSE loss across all batches (Python float).
    """
    model.train()
    device = torch.device(device)

    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        inp = batch["input"].to(device)       # (B, T_in, max_N)
        ctx = batch["context"].to(device)     # (B, T_in, 5)
        target = batch["target"].to(device)   # (B, 1, max_N) or (B, T_out, max_N)
        mask = batch["target_mask"].to(device)  # (B, 1, max_N)

        # AD-4: duck-type dispatch — spatial model gets 3 separate tensors;
        # HeadwayLSTM receives the flat concatenation as before (AD-1).
        if hasattr(model, "spatial") and model.spatial:
            input_mask = batch["input_mask"].to(device)  # (B, T_in, max_N) bool
            pred = model(inp, ctx, input_mask)
        else:
            # AD-1: concatenate headway + context before model forward.
            x = torch.cat([inp, ctx], dim=-1)  # (B, T_in, max_N + 5)
            pred = model(x)  # (B, max_N)

        # Squeeze time dimension from target/mask: (B, 1, max_N) → (B, max_N).
        target_sq = target.squeeze(1)  # (B, max_N)
        mask_sq = mask.squeeze(1)      # (B, max_N) bool

        loss = masked_mse_loss(pred, target_sq, mask_sq)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def evaluate_epoch(
    model: nn.Module,
    loader: Any,  # DataLoader or list of dict batches
    device: str | torch.device = "cpu",
) -> float:
    """Evaluate on val or test set for one epoch.

    Runs in eval mode with no gradient tracking (AC-TRAIN-3, AC-TRAIN-4).

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to evaluate.
    loader:
        Iterable yielding dicts with keys: input, context, target, target_mask.
    device:
        Device string or torch.device.

    Returns
    -------
    Mean masked MSE loss across all batches (Python float).
    """
    model.eval()
    device = torch.device(device)

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            inp = batch["input"].to(device)
            ctx = batch["context"].to(device)
            target = batch["target"].to(device)
            mask = batch["target_mask"].to(device)

            # AD-4: duck-type dispatch — spatial model gets 3 separate tensors;
            # HeadwayLSTM receives the flat concatenation as before (AD-1).
            if hasattr(model, "spatial") and model.spatial:
                input_mask = batch["input_mask"].to(device)  # (B, T_in, max_N) bool
                pred = model(inp, ctx, input_mask)
            else:
                # AD-1: concatenate headway + context before model forward.
                x = torch.cat([inp, ctx], dim=-1)
                pred = model(x)

            # Squeeze time dimension: (B, 1, max_N) → (B, max_N).
            target_sq = target.squeeze(1)
            mask_sq = mask.squeeze(1)

            loss = masked_mse_loss(pred, target_sq, mask_sq)
            total_loss += loss.item()
            n_batches += 1

    return total_loss / max(n_batches, 1)


# ---------------------------------------------------------------------------
# Early stopping
# ---------------------------------------------------------------------------

class EarlyStopping:
    """Monitor validation loss and stop training when no improvement (AD-9).

    Usage
    -----
    es = EarlyStopping(patience=10)
    for epoch in range(max_epochs):
        val_loss = evaluate_epoch(...)
        if es.step(val_loss, model):
            break
    # Restore best weights:
    model.load_state_dict(es.best_state_dict)

    Notes
    -----
    - Improvement is strictly less than the current best.
    - best_state_dict is a deep copy (AD-9, AC-TRAIN-6).
    - Counter resets on improvement; fires when counter > patience.
    """

    def __init__(self, patience: int) -> None:
        self._patience = patience
        self._counter = 0
        self._best_loss: float = float("inf")
        self._best_state_dict: dict[str, Any] | None = None
        self._should_stop: bool = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        """Process one validation epoch.

        Parameters
        ----------
        val_loss:
            Validation loss for this epoch.
        model:
            The model being trained; state_dict is deep-copied on improvement.

        Returns
        -------
        True if training should stop (patience exceeded), False otherwise.
        """
        if val_loss < self._best_loss:
            # Improvement: reset counter, save best state.
            self._best_loss = val_loss
            self._counter = 0
            self._best_state_dict = copy.deepcopy(model.state_dict())
        else:
            self._counter += 1
            if self._counter >= self._patience:
                self._should_stop = True

        return self._should_stop

    @property
    def should_stop(self) -> bool:
        """True if patience has been exceeded."""
        return self._should_stop

    @property
    def best_val_loss(self) -> float:
        """Lowest validation loss recorded so far (float('inf') if never improved)."""
        return self._best_loss

    @property
    def best_state_dict(self) -> dict[str, Any] | None:
        """Deep copy of the model state_dict from the best epoch.

        None if step() has never been called with an improving loss.
        """
        return self._best_state_dict


# ---------------------------------------------------------------------------
# Wave 3: GRID constant, train_model, grid_search, checkpoint I/O, denormalize
# ---------------------------------------------------------------------------

# Context vector dimensionality: hour_sin, hour_cos, dow_sin, dow_cos, atypical.
CONTEXT_DIM: int = 5

# Cartesian product: hidden ∈ {32,64,128} × layers ∈ {1,2} × dropout ∈ {0.0,0.2}
# × lr ∈ {1e-3,5e-4} = 24 configurations (AC-GRID-5).
GRID: list[TrainConfig] = [
    TrainConfig(hidden_size=h, num_layers=n, dropout=d, lr=lr)
    for h in [32, 64, 128]
    for n in [1, 2]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
]

# Fase 6a — Spatial grid (AD-6):
# conv_channels ∈ {1,8,16} × hidden ∈ {32,64} × layers ∈ {1,2}
# × dropout ∈ {0.0,0.2} × lr ∈ {1e-3,5e-4} = 3×2×2×2×2 = 48 configs.
SPATIAL_GRID: list[TrainConfig] = [
    TrainConfig(
        hidden_size=h,
        num_layers=n,
        dropout=d,
        lr=lr,
        conv_channels=c,
    )
    for c in [1, 8, 16]
    for h in [32, 64]
    for n in [1, 2]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
]

# Fase 6b — Transformer grid (obs #417 canonical grid):
# nhead ∈ {1,2} × d_model ∈ {16,32} × hidden ∈ {32,64}
# × dropout ∈ {0.0,0.2} × lr ∈ {1e-3,5e-4}, num_layers=1 fixed.
# All (nhead, d_model) combos satisfy d_model % nhead == 0:
#   nhead=1: 16%1==0, 32%1==0 ✓
#   nhead=2: 16%2==0, 32%2==0 ✓
# → 2×2×2×2×2 = 32 configs total (no filtering needed).
TRANSFORMER_GRID: list[TrainConfig] = [
    TrainConfig(
        hidden_size=h,
        num_layers=1,
        dropout=d,
        lr=lr,
        nhead=nh,
        d_model=dm,
    )
    for nh in [1, 2]
    for dm in [16, 32]
    for h in [32, 64]
    for d in [0.0, 0.2]
    for lr in [1e-3, 5e-4]
    if dm % nh == 0  # guard is always True for these values; kept for clarity
]


def train_model(
    model: nn.Module,
    train_dl: Any,
    val_dl: Any,
    config: TrainConfig,
    device: str | torch.device = "cpu",
) -> TrainResult:
    """Full training loop with early stopping (AC-TRAIN-7, AC-TRAIN-8).

    Steps per epoch:
    1. train_one_epoch → training loss.
    2. evaluate_epoch → validation loss.
    3. EarlyStopping.step: save best weights; break if patience exceeded.

    After training: restores best weights into model via load_state_dict.

    Parameters
    ----------
    model:
        HeadwayLSTM (or any nn.Module) to train.
    train_dl:
        Training dataloader (iterable of dicts).
    val_dl:
        Validation dataloader (iterable of dicts).
    config:
        TrainConfig controlling hyperparameters and training policy.
    device:
        Device string or torch.device.

    Returns
    -------
    TrainResult with best_val_loss, best_epoch, epochs_trained, and state_dict.
    """
    set_seed(config.seed)
    model.to(torch.device(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    early_stopping = EarlyStopping(patience=config.patience)

    train_losses: list[float] = []
    val_losses: list[float] = []
    best_epoch: int = 0
    epoch: int = 0

    for epoch in range(config.max_epochs):
        t_loss = train_one_epoch(model, train_dl, optimizer, device)
        v_loss = evaluate_epoch(model, val_dl, device)

        train_losses.append(t_loss)
        val_losses.append(v_loss)

        improved = v_loss < early_stopping.best_val_loss
        if improved:
            best_epoch = epoch

        if early_stopping.step(v_loss, model):
            break

    # Restore best weights (AD-9).
    if early_stopping.best_state_dict is not None:
        model.load_state_dict(early_stopping.best_state_dict)

    return TrainResult(
        best_val_loss=early_stopping.best_val_loss,
        best_epoch=best_epoch,
        epochs_trained=epoch + 1,
        train_losses=train_losses,
        val_losses=val_losses,
        state_dict=early_stopping.best_state_dict or {},
        config=config,
    )


def grid_search(
    train_dl: Any,
    val_dl: Any,
    max_N: int,
    configs: list[TrainConfig],
    device: str | torch.device = "cpu",
) -> list[TrainResult]:
    """Train one model per config; return results sorted ascending by best_val_loss (AC-GRID-1..3).

    Fase 6a: if config.conv_channels is not None, instantiates SpatialConvLSTM
    instead of HeadwayLSTM (AD-4/AD-5). The HeadwayLSTM path is unchanged.

    Parameters
    ----------
    train_dl:
        Training dataloader.
    val_dl:
        Validation dataloader.
    max_N:
        Maximum number of buses (determines model input_size = max_N + CONTEXT_DIM).
    configs:
        List of TrainConfig to evaluate. Use GRID for the full 24-config sweep,
        SPATIAL_GRID for the Fase 6a 48-config sweep.
    device:
        Device string or torch.device.

    Returns
    -------
    List of TrainResult, sorted ascending by best_val_loss (best config first).
    """
    results: list[TrainResult] = []

    for config in configs:
        # Fase 6b: nhead and conv_channels are mutually exclusive.
        if config.nhead is not None and config.conv_channels is not None:
            raise ValueError(
                f"nhead and conv_channels are mutually exclusive in TrainConfig. "
                f"Got nhead={config.nhead}, conv_channels={config.conv_channels}. "
                "Set one to None."
            )

        # Seed BEFORE instantiating the model so weight initialization is
        # controlled by config.seed (train_model re-seeds again before the
        # training loop). Without this, initial weights depend on ambient RNG
        # state and per-seed runs are not exactly reproducible.
        set_seed(config.seed)

        # Fase 6b: nhead-first dispatch → SpatialTransformer.
        if config.nhead is not None:
            model: nn.Module = SpatialTransformer(
                max_N=max_N,
                nhead=config.nhead,
                d_model=config.d_model,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        # Fase 6a: conv_channels → SpatialConvLSTM.
        elif config.conv_channels is not None:
            model = SpatialConvLSTM(
                max_N=max_N,
                conv_channels=config.conv_channels,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        # Default: HeadwayLSTM path (Fase 5).
        else:
            model = HeadwayLSTM(
                input_size=max_N + CONTEXT_DIM,
                hidden_size=config.hidden_size,
                output_size=max_N,
                num_layers=config.num_layers,
                dropout=config.dropout,
            )
        result = train_model(model, train_dl, val_dl, config, device)
        results.append(result)

    return sorted(results, key=lambda r: r.best_val_loss)


def save_checkpoint(result: TrainResult, path: Path) -> None:
    """Persist a TrainResult to disk as model.pt + config.json (AC-TRAIN-7).

    Parameters
    ----------
    result:
        TrainResult containing state_dict and config.
    path:
        Directory to create and write files into. Created if absent.

    Files written
    -------------
    model.pt     — torch state_dict (weights_only compatible).
    config.json  — serialised TrainConfig fields plus best_val_loss and epochs_trained.
    """
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    torch.save(result.state_dict, path / "model.pt")

    config_dict: dict[str, Any] = {
        "hidden_size": result.config.hidden_size,
        "num_layers": result.config.num_layers,
        "dropout": result.config.dropout,
        "lr": result.config.lr,
        "batch_size": result.config.batch_size,
        "max_epochs": result.config.max_epochs,
        "patience": result.config.patience,
        "seed": result.config.seed,
        "best_val_loss": result.best_val_loss,
        "epochs_trained": result.epochs_trained,
    }
    # AD-9 / AD-5: persist conv_channels only when it is set.
    if result.config.conv_channels is not None:
        config_dict["conv_channels"] = result.config.conv_channels
    # Fase 6b: persist nhead and d_model only when set (mirror conv_channels pattern).
    if result.config.nhead is not None:
        config_dict["nhead"] = result.config.nhead
    if result.config.d_model is not None:
        config_dict["d_model"] = result.config.d_model
    (path / "config.json").write_text(json.dumps(config_dict, indent=2))


def load_checkpoint(
    path: Path, max_N: int
) -> tuple[HeadwayLSTM | SpatialConvLSTM | SpatialTransformer, TrainConfig]:
    """Restore a model and its TrainConfig from a checkpoint directory (AC-TRAIN-7).

    Fase 6a (AD-9): if config.json contains a ``conv_channels`` key, instantiates
    SpatialConvLSTM; otherwise falls back to HeadwayLSTM (backward compatible).
    Fase 6b: if config.json contains a ``nhead`` key, instantiates SpatialTransformer
    (nhead-first priority, checked before conv_channels).

    Parameters
    ----------
    path:
        Directory previously written by save_checkpoint.
    max_N:
        Maximum number of buses; used to reconstruct model input_size and output_size.

    Returns
    -------
    (model, config) where model weights match the saved state_dict.
    """
    path = Path(path)
    config_dict = json.loads((path / "config.json").read_text())

    conv_channels: int | None = config_dict.get("conv_channels", None)
    nhead: int | None = config_dict.get("nhead", None)
    d_model: int | None = config_dict.get("d_model", None)

    config = TrainConfig(
        hidden_size=config_dict["hidden_size"],
        num_layers=config_dict["num_layers"],
        dropout=config_dict["dropout"],
        lr=config_dict["lr"],
        batch_size=config_dict.get("batch_size", 32),
        max_epochs=config_dict.get("max_epochs", 50),
        patience=config_dict.get("patience", 10),
        seed=config_dict.get("seed", 42),
        conv_channels=conv_channels,
        nhead=nhead,
        d_model=d_model,
    )

    # Fase 6b: nhead-first model class selection.
    model: HeadwayLSTM | SpatialConvLSTM | SpatialTransformer
    if nhead is not None:
        model = SpatialTransformer(
            max_N=max_N,
            nhead=nhead,
            d_model=d_model,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    elif conv_channels is not None:
        model = SpatialConvLSTM(
            max_N=max_N,
            conv_channels=conv_channels,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    else:
        model = HeadwayLSTM(
            input_size=max_N + CONTEXT_DIM,
            hidden_size=config.hidden_size,
            output_size=max_N,
            num_layers=config.num_layers,
            dropout=config.dropout,
        )
    state_dict = torch.load(path / "model.pt", weights_only=True)
    model.load_state_dict(state_dict)
    return model, config


def denormalize_predictions(
    pred: torch.Tensor,
    mean: float,
    std: float,
) -> torch.Tensor:
    """Convert z-scored predictions back to original scale (minutes) (AD-6, AC-EVAL-1).

    Formula: pred_minutes = pred_z * (std + 1e-8) + mean

    Parameters
    ----------
    pred:
        Tensor of z-scored predictions from the model.
    mean:
        Per-corridor mean (in minutes) used during z-scoring.
    std:
        Per-corridor standard deviation (in minutes) used during z-scoring.

    Returns
    -------
    Tensor of predicted headways in minutes (same shape as pred).
    """
    return pred * (std + 1e-8) + mean

## `CONTEXT_DIM` 5 → 4

`train.py` dimensiona el modelo como `max_N + CONTEXT_DIM`. Al retirar la bandera
de día atípico el contexto pasa de 5 a 4 columnas, así que la constante se
rebindea **antes** de construir cualquier modelo. En el espacio de nombres plano
del notebook, `grid_search` y `train_model` leen esta misma global.

In [ ]:

CTX_COLS = list(CAUSAL_CONTEXT_FEATURE_NAMES)
assert "atypical_flag" not in CTX_COLS, "the leaking flag must not come back"

CONTEXT_DIM = len(CTX_COLS)
assert CONTEXT_DIM == 4, CONTEXT_DIM
print(f"CONTEXT_DIM rebound to {CONTEXT_DIM}: {CTX_COLS}")

## Cargar, partir, winsorizar, normalizar, contexto

Idéntico al pipeline anterior salvo por el contexto: se codifican las 5 columnas
pero solo se consumen las 4 causales.

El corte temporal se resuelve por **nombre** contra el módulo `splits` embebido.
Un nombre desconocido levanta `KeyError` acá mismo, antes de tocar la GPU: caer
en silencio al corte publicado produciría resultados etiquetados como un origen
y calculados sobre otro.

In [ ]:

FOLD = fold_by_name(FOLD_NAME)
print(f"Fold {FOLD.name}: train {FOLD.train_start}..{FOLD.train_end} "
      f"({FOLD.train_days}d) | val {FOLD.val_start}..{FOLD.val_end} "
      f"({FOLD.val_days}d) | test {FOLD.test_start}..{FOLD.test_end} "
      f"({FOLD.test_days}d)")

RAW = {}
RAW["E4"] = pl.read_parquet(_resolve_input("headways_E4.parquet")).with_columns(pl.lit(4, dtype=pl.Int64).alias("empresaid"))
for name, frame in RAW.items():
    print(f"{name}: {frame.height:,} rows")

PREPARED = {}
STATS = {}
for name, frame in RAW.items():
    df_split = split_temporal(frame, FOLD)
    df_winsor, threshold = winsorize_train_p99(df_split)
    stats = compute_normalization_stats(df_winsor.filter(pl.col("split") == "train"))
    df_z = apply_zscore(df_winsor, stats)
    # No atypical calendar is passed: the flag column is emitted as all-zero and
    # then never selected (CTX_COLS excludes it).
    PREPARED[name] = encode_context(df_z)
    STATS[name] = stats
    print(f"{name}: winsor threshold={threshold:.4f} min, "
          f"splits={df_split.group_by('split').agg(pl.len()).sort('split').to_dicts()}")

## Índice canónico + portón de población compartida

Construye el índice para h=5 en cada split y **verifica su SHA-256**
contra `sample_index_manifest.csv`. Si no coincide, la corrida se detiene antes
de entrenar: significa que el código o los bytes de entrada cambiaron y que esta
corrida ya no es comparable con el refit del XGBoost.

In [ ]:

T_IN = DEFAULT_T_IN   # 12
BATCH_SIZE = 128

def _index_digest(index: pl.DataFrame) -> str:
    canonical = index.select(
        ["empresaid", "direction", "start_ts", "target_ts", "horizon"]
    ).sort(["empresaid", "direction", "horizon", "start_ts"])
    payload = canonical.write_csv(datetime_format="%Y-%m-%dT%H:%M:%S")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

INDEX = {}
for name, df in PREPARED.items():
    for split in ["train", "val", "test"]:
        part = df.filter(pl.col("split") == split)
        idx = make_sample_index(part, horizon=HORIZON, T_in=T_IN)
        digest = _index_digest(idx)
        expected = INDEX_DIGESTS[f"{name}|{split}"]
        if digest != expected:
            raise ValueError(
                f"SHARED-POPULATION GATE FAILED for {name}/{split}/h{HORIZON}: "
                f"digest {digest} != frozen {expected}. This run is NOT comparable "
                f"with the XGBoost refit — stop and re-freeze the manifest."
            )
        INDEX[(name, split)] = idx
        print(f"  {name}/{split}: {idx.height:,} samples  digest OK")
print("\nShared-population gate: PASSED for every split.")

## Materialización

`materialize_arrays` vive en `data/contiguous_dataset` y está cubierto por tests
que verifican que coincide muestra por muestra con el `Dataset`. El notebook no
lleva su propia copia.

In [ ]:

import torch
import time as _time

MAX_N = {}
BATCHES = {}
TEST_ARRAYS = {}

def _to_batches(arrays: dict) -> list:
    n = arrays["input"].shape[0]
    tensors = {k: torch.from_numpy(v) for k, v in arrays.items()}
    return [
        {k: v[s:min(s + BATCH_SIZE, n)] for k, v in tensors.items()}
        for s in range(0, n, BATCH_SIZE)
    ]

for name, df in PREPARED.items():
    max_n_by_dir = compute_max_N(df.filter(pl.col("split") == "train"), quantile=0.99)
    global_max_N = max(max_n_by_dir.values())
    MAX_N[name] = global_max_N
    print(f"\n{name}: max_N per direction={max_n_by_dir}, global={global_max_N}")

    loaders = {}
    for split in ["train", "val"]:
        t0 = _time.time()
        arrays = materialize_arrays(
            df.filter(pl.col("split") == split), INDEX[(name, split)],
            max_N=global_max_N, T_in=T_IN, horizon=HORIZON, context_cols=tuple(CTX_COLS),
        )
        loaders[split] = _to_batches(arrays)
        print(f"  {name} {split}: {arrays['input'].shape[0]:,} samples -> "
              f"{len(loaders[split]):,} batches in {_time.time()-t0:.1f}s")
    BATCHES[name] = loaders

    # Test is kept per-direction: denormalization uses direction-specific stats.
    test_df = df.filter(pl.col("split") == "test")
    per_dir = {}
    for direction in [-1, 1]:
        sub_idx = INDEX[(name, "test")].filter(pl.col("direction") == direction)
        arrays = materialize_arrays(
            test_df.filter(pl.col("direction") == direction), sub_idx,
            max_N=global_max_N, T_in=T_IN, horizon=HORIZON, context_cols=tuple(CTX_COLS),
        )
        per_dir[direction] = (sub_idx, arrays, _to_batches(arrays))
        print(f"  {name} test dir={direction:+d}: {arrays['input'].shape[0]:,} samples")
    TEST_ARRAYS[name] = per_dir

## Entrenamiento

Se reusa la configuración ganadora congelada por corredor. E4 nunca tuvo una
ganadora congelada, así que conserva su mini-grid de 3 configuraciones
seleccionadas **solo sobre validación**.

In [ ]:

CONFIGS = {
    "E4": [TrainConfig(hidden_size=32, num_layers=1, dropout=0.0, lr=5e-4),
     TrainConfig(hidden_size=32, num_layers=2, dropout=0.2, lr=5e-4),
     TrainConfig(hidden_size=64, num_layers=1, dropout=0.0, lr=5e-4)],
}

RESULTS = {}
for name, loaders in BATCHES.items():
    print(f"\n{name} training: max_N={MAX_N[name]}, device={DEVICE}")
    res = grid_search(
        train_dl=loaders["train"],
        val_dl=loaders["val"],
        max_N=MAX_N[name],
        configs=CONFIGS[name],
        device=DEVICE,
    )
    best = res[0]
    print(f"  {name} winner: hidden={best.config.hidden_size}, "
          f"layers={best.config.num_layers}, dropout={best.config.dropout}, "
          f"lr={best.config.lr}, val_loss={best.best_val_loss:.6f} "
          f"(epoch {best.best_epoch}, of {len(res)} configs)")
    RESULTS[name] = res

## Evaluación + residuos con clave completa

La persistencia (B1) es el último paso observado de la ventana, así que se
compara contra **exactamente las mismas muestras** — pareado por construcción.
Los residuos salen con la clave completa, de modo que agrupar por día de servicio
o construir un perfil por posición del vector ya no exige otra corrida.

In [ ]:

def evaluate_corridor(name):
    best = RESULTS[name][0]
    stats = STATS[name]
    empresa_id = int(list(stats.means.keys())[0][0])
    max_N = MAX_N[name]

    model = HeadwayLSTM(
        input_size=max_N + CONTEXT_DIM,
        hidden_size=best.config.hidden_size,
        output_size=max_N,
        num_layers=best.config.num_layers,
        dropout=best.config.dropout,
    )
    model.load_state_dict(best.state_dict)
    model.eval()
    model.to(torch.device(DEVICE))

    rows, resid_frames = [], []
    pooled = {"pred": [], "true": []}

    for direction in [-1, 1]:
        sub_idx, arrays, batches = TEST_ARRAYS[name][direction]
        mean_val = stats.means[(empresa_id, direction)]
        std_val = stats.stds[(empresa_id, direction)]

        preds = []
        with torch.no_grad():
            for batch in batches:
                x = torch.cat([batch["input"].to(DEVICE), batch["context"].to(DEVICE)], dim=-1)
                preds.append(
                    denormalize_predictions(model(x), mean_val, std_val).cpu().numpy()
                )
        pred_min = np.concatenate(preds) if preds else np.zeros((0, max_N), dtype=np.float32)

        target_min = denormalize_predictions(
            torch.from_numpy(arrays["target"][:, 0]), mean_val, std_val
        ).numpy()
        persist_min = denormalize_predictions(
            torch.from_numpy(arrays["input"][:, T_IN - 1, :]), mean_val, std_val
        ).numpy()
        tmask = arrays["target_mask"][:, 0]
        pmask = arrays["input_mask"][:, T_IN - 1, :]

        valid = tmask
        mae_val = mae(target_min[valid], pred_min[valid])
        rmse_val = rmse(target_min[valid], pred_min[valid])
        print(f"{name} dir={direction:+d}: MAE={mae_val:.4f} RMSE={rmse_val:.4f} "
              f"(n_valid={int(valid.sum()):,})")
        pooled["pred"].append(pred_min[valid])
        pooled["true"].append(target_min[valid])

        dir_str = f"+{direction}" if direction > 0 else str(direction)
        for metric_name, metric_val in [("MAE", mae_val), ("RMSE", rmse_val)]:
            rows.append({"corridor": name, "direction": dir_str, "baseline": "LSTM_CONTIG",
                         "metric": metric_name, "value": float(metric_val), "horizon": HORIZON})

        resid_frames.append(build_keyed_residuals(
            sub_idx, corridor=name, split="test",
            y_true=target_min, y_pred_model=pred_min, y_pred_persist=persist_min,
            target_mask=tmask, persist_mask=pmask,
        ))

    all_pred = np.concatenate(pooled["pred"])
    all_true = np.concatenate(pooled["true"])
    agg_mae, agg_rmse = mae(all_true, all_pred), rmse(all_true, all_pred)
    print(f"{name} aggregate: MAE={agg_mae:.4f} RMSE={agg_rmse:.4f} (n={len(all_pred):,})")
    for metric_name, metric_val in [("MAE", agg_mae), ("RMSE", agg_rmse)]:
        rows.append({"corridor": name, "direction": "aggregate", "baseline": "LSTM_CONTIG",
                     "metric": metric_name, "value": float(metric_val), "horizon": HORIZON})

    return rows, pl.concat(resid_frames)

all_rows, all_resid = [], []
for name in PREPARED:
    r, res = evaluate_corridor(name)
    all_rows.extend(r)
    all_resid.append(res)

results = pl.DataFrame(all_rows)
results.write_csv(RESULTS_OUT)
print(f"\nResults written: {RESULTS_OUT} ({results.height} rows)")
print(results)

residuals = pl.concat(all_resid)
assert_key_is_unique(residuals)
residuals.write_csv(RESID_OUT)
print(f"Residuals written: {RESID_OUT} ({residuals.height:,} rows, key verified unique)")

## Comparación con los baselines del pipeline anterior

⚠️ **Lectura con cuidado.** Los baselines vienen del pipeline **anterior**, con
otra población. La comparación es orientativa; la cifra canónica sale del refit
del XGBoost sobre el índice compartido.

In [ ]:

baselines_csv = _find_baselines_csv()
if baselines_csv is not None:
    baselines = pl.read_csv(baselines_csv).filter(pl.col("horizon") == HORIZON)
    print(f"Baselines (OLD population) from {baselines_csv}: {baselines.height} rows")
    print(pl.concat([
        baselines.filter(pl.col("metric") == "MAE"),
        results.filter(pl.col("metric") == "MAE"),
    ]).sort(["corridor", "direction", "baseline"]))
    print("\nNOTE: baselines above ran on the OLD population — orientation only.")
else:
    print("Baselines CSV not found — skipping the orientation comparison.")